In [1]:
# Upload kaggle.json manually when prompted
from google.colab import files
files.upload()

# Move kaggle.json to the correct folder
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API authenticated!")

Saving kaggle.json to kaggle.json
Kaggle API authenticated!


In [2]:
# Create datasets folder
!mkdir -p /content/datasets

# Download the VITON-HD dataset (replace dataset name if needed)
!kaggle datasets download -d marquis03/high-resolution-viton-zalando-dataset -p /content/datasets

Dataset URL: https://www.kaggle.com/datasets/marquis03/high-resolution-viton-zalando-dataset
License(s): CC-BY-NC-SA-4.0


In [3]:
# Unzip it
!unzip -q /content/datasets/high-resolution-viton-zalando-dataset.zip -d /content/datasets/viton_hd

print("VITON-HD dataset downloaded and extracted!")

VITON-HD dataset downloaded and extracted!


In [4]:
!pip install ftfy regex
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-sk2sf_30
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-sk2sf_30
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
import json
import numpy as np
import torch
import clip
from torchvision import transforms
from torchvision.models import resnet50
from sklearn.metrics import accuracy_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

In [6]:
# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/datasets/viton_hd/test"
TEST_PAIRS = "/content/datasets/viton_hd/test_pairs.txt"
SAVE_DIR = "artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

In [7]:
# --- Helper Functions ---
def load_image(path):
    return Image.open(path).convert('RGB')

def load_openpose_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    keypoints = np.array(data['people'][0]['pose_keypoints_2d']).reshape(-1, 3)
    return keypoints

def extract_clip_features(model, preprocess, image):
    image_input = preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        features = model.encode_image(image_input)
    return features.cpu().numpy().flatten()

def extract_resnet_features(model, image):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
    ])
    image_tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        features = model(image_tensor)
    return features.cpu().numpy().flatten()

def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

def retrieve_topk(query_feat, gallery_feats, gallery_paths, topk=5):
    sims = [cosine_similarity(query_feat, feat) for feat in gallery_feats]
    topk_idx = np.argsort(sims)[::-1][:topk]
    return [gallery_paths[i] for i in topk_idx]

def estimate_body_shape(keypoints):
    torso_length = np.linalg.norm(keypoints[1][:2] - keypoints[8][:2])
    hip_width = np.linalg.norm(keypoints[11][:2] - keypoints[8][:2])
    ratio = hip_width / torso_length
    if ratio > 0.7:
        return "pear"
    elif ratio < 0.5:
        return "inverted"
    else:
        return "rectangle"

def visualize_features(features, labels, save_path):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    reduced = tsne.fit_transform(features)
    plt.figure(figsize=(10,7))
    for label in np.unique(labels):
        idx = np.where(labels == label)
        plt.scatter(reduced[idx,0], reduced[idx,1], label=label)
    plt.legend()
    plt.savefig(save_path)
    plt.close()

In [8]:
# --- Loading Models ---
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
resnet_model = resnet50(pretrained=True).to(DEVICE)
resnet_model.eval()

# --- Preparing Dataset ---
with open(TEST_PAIRS, 'r') as f:
    lines = f.readlines()
pairs = [line.strip().split() for line in lines]

person_images = [p[0] for p in pairs]
cloth_images = [p[1] for p in pairs]

# Precompute Clothing Features
cloth_dir = os.path.join(DATASET_PATH, "cloth")
cloth_paths = sorted([os.path.join(cloth_dir, c) for c in os.listdir(cloth_dir) if c.endswith('.jpg')])
cloth_clip_feats = []
for p in tqdm(cloth_paths, desc="Extracting CLIP features for clothes"):
    cloth_clip_feats.append(extract_clip_features(clip_model, clip_preprocess, load_image(p)))

100%|███████████████████████████████████████| 338M/338M [00:08<00:00, 39.8MiB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s]
Extracting CLIP features for clothes: 100%|██████████| 2032/2032 [00:49<00:00, 41.45it/s]


In [16]:
# 📊 Retrieval Metrics Functions
from sklearn.metrics import average_precision_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Retrieval Metrics
def precision_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted = predicted[:k]
    return len(set(predicted) & actual_set) / len(predicted)

def recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted = predicted[:k]
    return len(set(predicted) & actual_set) / len(actual_set)

def mean_average_precision(actual_list, predicted_list, k=5):
    average_precisions = []
    for actual, predicted in zip(actual_list, predicted_list):
        ap = 0.0
        correct = 0
        for i in range(min(k, len(predicted))):
            if predicted[i] in actual:
                correct += 1
                ap += correct / (i + 1)
        if len(actual) > 0:
            average_precisions.append(ap / min(len(actual), k))
    return sum(average_precisions) / len(average_precisions)

def compute_retrieval_metrics(actual_labels, predicted_labels, k_list=[1,3,5]):
    for k in k_list:
        precisions = [precision_at_k(a, p, k) for a, p in zip(actual_labels, predicted_labels)]
        recalls = [recall_at_k(a, p, k) for a, p in zip(actual_labels, predicted_labels)]

        print(f"Precision@{k}: {sum(precisions)/len(precisions):.4f}")
        print(f"Recall@{k}: {sum(recalls)/len(recalls):.4f}")

    map_score = mean_average_precision(actual_labels, predicted_labels, k=max(k_list))
    print(f"Mean Average Precision (mAP@{max(k_list)}): {map_score:.4f}")

# t-SNE Evaluation Metrics
def evaluate_tsne_clusters(tsne_embeddings, n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(tsne_embeddings)

    silhouette = silhouette_score(tsne_embeddings, cluster_labels)
    davies_bouldin = davies_bouldin_score(tsne_embeddings, cluster_labels)
    calinski_harabasz = calinski_harabasz_score(tsne_embeddings, cluster_labels)

    print(f"Silhouette Score: {silhouette:.4f}")
    print(f"Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
    print(f"Calinski-Harabasz Score: {calinski_harabasz:.4f} (higher is better)")

In [17]:
# --- Evaluation ---
clip_top1, clip_top3, clip_top5 = [], [], []
clip_top1_shapeaware, clip_top3_shapeaware, clip_top5_shapeaware = [], [], []

body_shapes = {}
cloth_name_to_shape = {}

# Lists for Retrieval Metrics
all_actual_labels = []
all_predicted_labels = []

# Estimating body shapes for all clothing owners
for cloth_path in tqdm(cloth_paths, desc="Estimating body shapes for clothes"):
    cloth_name = os.path.basename(cloth_path)
    person_name = cloth_name  # assuming 1-to-1 mapping
    json_path = os.path.join(DATASET_PATH, "openpose_json", person_name.replace('.jpg', '_keypoints.json'))
    if os.path.exists(json_path):
        keypoints = load_openpose_json(json_path)
        shape = estimate_body_shape(keypoints)
        cloth_name_to_shape[cloth_name] = shape

# Main evaluation loop
for person_img_name, gt_cloth_name in tqdm(pairs, desc="Processing test pairs"):
    person_path = os.path.join(DATASET_PATH, "image", person_img_name)
    person_img = load_image(person_path)

    person_clip_feat = extract_clip_features(clip_model, clip_preprocess, person_img)

    # Body shape estimation
    json_path = os.path.join(DATASET_PATH, "openpose_json", person_img_name.replace('.jpg', '_keypoints.json'))
    keypoints = load_openpose_json(json_path)
    shape = estimate_body_shape(keypoints)
    body_shapes[person_img_name] = shape

    # Normal Retrieval
    retrieved = retrieve_topk(person_clip_feat, cloth_clip_feats, cloth_paths, topk=5)
    retrieved_names = [os.path.basename(p) for p in retrieved]

    clip_top1.append(gt_cloth_name == retrieved_names[0])
    clip_top3.append(gt_cloth_name in retrieved_names[:3])
    clip_top5.append(gt_cloth_name in retrieved_names[:5])

    # Collect for retrieval metrics
    all_actual_labels.append([gt_cloth_name])  # ground-truth wrapped in a list
    all_predicted_labels.append(retrieved_names)

    # Shape-Aware Retrieval
    valid_idxs = [i for i, p in enumerate(cloth_paths) if cloth_name_to_shape.get(os.path.basename(p), None) == shape]
    if valid_idxs:
        shape_aware_feats = [cloth_clip_feats[i] for i in valid_idxs]
        shape_aware_paths = [cloth_paths[i] for i in valid_idxs]
        retrieved_shapeaware = retrieve_topk(person_clip_feat, shape_aware_feats, shape_aware_paths, topk=5)
        retrieved_names_shapeaware = [os.path.basename(p) for p in retrieved_shapeaware]

        clip_top1_shapeaware.append(gt_cloth_name == retrieved_names_shapeaware[0])
        clip_top3_shapeaware.append(gt_cloth_name in retrieved_names_shapeaware[:3])
        clip_top5_shapeaware.append(gt_cloth_name in retrieved_names_shapeaware[:5])


Processing test pairs: 100%|██████████| 2032/2032 [04:43<00:00,  7.16it/s]


In [18]:
# --- Metrics Calculation ---

def compute_accuracy(lst):
    return np.mean(lst) * 100 if lst else 0

clip_results = {
    "Top-1": compute_accuracy(clip_top1),
    "Top-3": compute_accuracy(clip_top3),
    "Top-5": compute_accuracy(clip_top5),
}

clip_results_shapeaware = {
    "Top-1": compute_accuracy(clip_top1_shapeaware),
    "Top-3": compute_accuracy(clip_top3_shapeaware),
    "Top-5": compute_accuracy(clip_top5_shapeaware),
}

with open(os.path.join(SAVE_DIR, "clip_results.json"), 'w') as f:
    json.dump(clip_results, f, indent=4)

with open(os.path.join(SAVE_DIR, "clip_results_shapeaware.json"), 'w') as f:
    json.dump(clip_results_shapeaware, f, indent=4)

with open(os.path.join(SAVE_DIR, "body_shapes.json"), 'w') as f:
    json.dump(body_shapes, f, indent=4)

# --- t-SNE Visualization ---
feature_stack = np.vstack(cloth_clip_feats)
labels = np.array(["Cloth"] * len(cloth_clip_feats))
visualize_features(feature_stack, labels, os.path.join(SAVE_DIR, "clip_tsne.png"))

print("All artifacts generated and saved to:", SAVE_DIR)

All artifacts generated and saved to: artifacts


In [19]:
# --- Retrieval Metrics Calculation ---
print("\nRetrieval Evaluation Metrics:")
compute_retrieval_metrics(all_actual_labels, all_predicted_labels)


Retrieval Evaluation Metrics:
Precision@1: 0.0005
Recall@1: 0.0005
Precision@3: 0.0003
Recall@3: 0.0010
Precision@5: 0.0003
Recall@5: 0.0015
Mean Average Precision (mAP@5): 0.0008


In [20]:
# --- Cluster Evaluation Metrics ---
print("\nt-SNE Cluster Evaluation Metrics:")
evaluate_tsne_clusters(feature_stack)


t-SNE Cluster Evaluation Metrics:
Silhouette Score: 0.0479
Davies-Bouldin Index: 3.0245 (lower is better)
Calinski-Harabasz Score: nan (higher is better)


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/cluster/_unsupervised.py:386: RuntimeWarning: overflow encountered in scalar multiply
  else extra_disp * (n_samples - n_labels) / (intra_disp * (n_labels - 1.0))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/cluster/_unsupervised.py:386: RuntimeWarning: invalid value encountered in scalar divide
  else extra_disp * (n_samples - n_labels) / (intra_disp * (n_labels - 1.0))
